# 03 モデルの妥当性チェック

`python -m ct125 score` の出力 `data/processed/<ride_id>/scored.parquet`（`z_speed`、`score_if`）について、docs/analysis_policy.md 5. の 3 項目を確認する。
精度（precision 等）は計算しない：正解ラベルがないので意味を持たない。ここで見るのは「筋が通っているか」だけ。

- `z_speed`：速度ビン（5 km/h）ごとの `log_rms` の中央値・MAD からの逸脱。走行中（≥5 km/h）の窓だけ `is_anomaly_z` の判定対象。
- `score_if`：振動特徴量だけで学習した Isolation Forest（速度は使わない。GPS なし走行にも付く）。

緯度経度の実値は表示しない（グリッド集計は ride_004 の最初の fix を原点にした相対座標で行う）。

In [ ]:
import json, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
plt.rcParams["font.family"] = ["Meiryo", "Yu Gothic", "MS Gothic", "sans-serif"]; plt.rcParams["axes.unicode_minus"] = False
PROC = Path("../data/processed")
S = pd.concat([pd.read_parquet(p) for p in sorted(PROC.glob("ride_*/scored.parquet"))], ignore_index=True)
BL = json.load(open(sorted(glob.glob("../data/models/*/baseline.json"))[-1], encoding="utf-8"))
M = S[~S.is_edge]                       # 学習・評価は先頭末尾 60 秒を除く
MV = M[M.has_gps & ~M.is_stopped]       # 走行中かつ GPS あり
S.groupby("ride_id").agg(windows=("win","size"), z_available=("z_speed", lambda s: s.notna().sum()), z_gt3=("is_anomaly_z","sum"),
                         if_anomaly=("is_anomaly_if","sum"), z_max=("z_speed","max"), z_min=("z_speed","min")).round(2)

## 1. 速度との関係：ベースラインの中央値と ±3 MAD の帯

In [ ]:
bins = pd.DataFrame(BL["bins"])
fig, ax = plt.subplots(figsize=(12, 6))
for rid, c in [("ride_004", "tab:blue"), ("ride_005", "tab:red"), ("ride_003", "tab:gray")]:
    g = M[(M.ride_id == rid) & M.has_gps]
    ax.scatter(g.speed_kmh, g.log_rms, s=7, alpha=.5, color=c, label=rid)
mid = (bins.lo + bins.hi) / 2
ax.step(bins.lo, bins["median"], where="post", color="k", lw=1.5, label="ビン中央値")
ax.fill_between(np.r_[bins.lo, bins.hi.iloc[-1]], np.r_[bins["median"] - 3 * 1.4826 * bins["mad"], (bins["median"] - 3 * 1.4826 * bins["mad"]).iloc[-1]],
                np.r_[bins["median"] + 3 * 1.4826 * bins["mad"], (bins["median"] + 3 * 1.4826 * bins["mad"]).iloc[-1]], step="post", color="k", alpha=.08, label="±3 MAD（z=±3）")
for _, b in bins.iterrows():
    ax.text((b.lo + b.hi) / 2, -3.9, str(b.n), ha="center", fontsize=7, color="gray")
ax.axvline(5, color="gray", ls=":", lw=.8); ax.text(5.5, -3.6, "停止判定 5 km/h", fontsize=8, color="gray")
ax.set_xlabel("speed_kmh"); ax.set_ylabel("log_rms"); ax.set_title("速度ビンごとの log_rms（下の数字＝ビン内の学習窓数。60 km/h 超は窓が少なく 55〜60 の値を借りている）"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
print("走行中（≥5 km/h、is_edge 除く）の窓:", len(MV), " z>3:", int((MV.z_speed > 3).sum()), " z<-3:", int((MV.z_speed < -3).sum()), " z の最大:", round(MV.z_speed.max(), 2))

- 速度が上がるほど中央値が上がる（5 km/h 以上で単調に近い）。期待どおり。
- 0〜5 km/h のビンはアイドリングの窓が大半で MAD が極端に小さい（0.057）ため、発進・取り回しの窓がすべて z>3 になる。そのため `is_anomaly_z` は走行中の窓だけに立てる。
- **走行中に z>3 の窓は 1 つもない**（最大 z ≈ 2.1）。健康な車両・同じ道の往復・1 日分のデータでは、ベースラインから 3 MAD 以上離れる窓はなかった、という結果。しきい値は `config/default.yaml` の `z_threshold` で変えられるが、ここでは下げない（下げる根拠がない）。
- z<−3（その速度にしては振動が小さすぎる）が 23 窓あり、すべて ride_004 14:15:40〜49 の「98 km/h」区間。GPS の速度解の誤りを振動側からも裏づける。

## 2. 往路・復路の一致：約 20 m グリッドで rms の中央値を比べる

In [ ]:
GRID_M = 20
g4 = S[(S.ride_id == "ride_004") & S.has_gps & ~S.is_stopped]
g5 = S[(S.ride_id == "ride_005") & S.has_gps & ~S.is_stopped]
lat0, lon0 = g4.lat.iloc[0], g4.lon.iloc[0]        # 原点（表示しない）
def grid(df):
    x = (df.lon - lon0) * 111320 * np.cos(np.deg2rad(lat0)); y = (df.lat - lat0) * 110540
    return df.assign(gx=np.floor(x / GRID_M).astype(int), gy=np.floor(y / GRID_M).astype(int))
c4 = grid(g4).groupby(["gx", "gy"]).rms.median().rename("rms_out")
c5 = grid(g5).groupby(["gx", "gy"]).rms.median().rename("rms_back")
both = pd.concat([c4, c5], axis=1).dropna()
rho, p = spearmanr(both.rms_out, both.rms_back)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(both.rms_out, both.rms_back, s=12, alpha=.6); lim = [0, max(both.max()) * 1.05]
ax.plot(lim, lim, "k--", lw=.8); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("往路 ride_004 の rms 中央値 [g]"); ax.set_ylabel("復路 ride_005 の rms 中央値 [g]")
ax.set_title(f"同じ {GRID_M} m グリッドを通った {len(both)} マス：Spearman ρ = {rho:.2f}（p = {p:.1e}）"); ax.grid(alpha=.3)
plt.tight_layout()
print(f"往路のマス {len(c4)}, 復路のマス {len(c5)}, 共通 {len(both)}, Spearman {rho:.3f}")

In [ ]:
# グリッド幅を変えても結論が変わらないか（rms・z_speed・速度の 3 つ）
rows = []
for G in [20, 30, 50, 100]:
    def grid_g(df):
        x = (df.lon - lon0) * 111320 * np.cos(np.deg2rad(lat0)); y = (df.lat - lat0) * 110540
        return df.assign(gx=np.floor(x / G).astype(int), gy=np.floor(y / G).astype(int))
    for col in ["rms", "z_speed", "speed_kmh"]:
        a = grid_g(g4).groupby(["gx", "gy"])[col].median(); b = grid_g(g5).groupby(["gx", "gy"])[col].median()
        bb = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
        r, p = spearmanr(bb.a, bb.b)
        rows.append(dict(grid_m=G, 列=col, 往路マス=len(a), 復路マス=len(b), 共通マス=len(bb), spearman=round(r, 3), p=round(p, 3)))
pd.DataFrame(rows)

**結果：往路・復路の一致は確認できなかった。** 20〜100 m のどのグリッド幅でも rms の相関はほぼ 0（p > 0.3）。

考えられる理由：
- 往路 233 マスのうち復路と共通なのは 56 マスだけ（20 m）。対向車線は 7〜10 m 離れ、GPS の位置誤差も数 m あるので、同じ道でも同じマスに入らない。往復で経路自体が違う区間もあると思われる。
- 同じマスでの **速度** も往復で一致しない（Spearman ≈ 0）。信号・右左折の多い市街地では、同じ場所でも通過速度が毎回違う。
- ロガーは置いただけの暫定固定で、路面以外の揺れ（置き方の共振）が混ざる。
- 1 日・1 往復分のデータでは、場所の効果を速度・運転の効果から分けるには足りない。

このチェックは「この走行データでは場所に由来する振動は確認できなかった」と正直に書く。今後は同じ道を同じ車線で複数回走る（往復ではなく周回）、ネジ止めで固定する、で再確認する。

## 3. 上位異常の目視：z_speed 上位 10 窓（走行中）

In [ ]:
cols = ["ride_id", "t_abs", "t_rel", "speed_kmh", "rms", "p2p", "crest", "band_1_5", "band_5_15", "band_15_40", "z_speed", "score_if", "is_anomaly_if"]
top = MV.nlargest(10, "z_speed")[cols].copy()
top["t_abs"] = top.t_abs.dt.strftime("%H:%M:%S")
top.round(2).reset_index(drop=True)

In [ ]:
# 上位窓の前後 30 秒：突発（1 窓だけ高い）か、区間（数窓つづく）か
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
for ax, (rid, t) in zip(axes, [("ride_004", 250.5), ("ride_004", 173.5)]):
    g = S[(S.ride_id == rid) & (S.t_rel > t - 30) & (S.t_rel < t + 30)]
    ax.plot(g.t_abs, g.rms, "-o", ms=3, label="rms"); ax.set_ylabel("rms [g]")
    ax2 = ax.twinx(); ax2.plot(g.t_abs, g.speed_kmh, color="tab:green", lw=.8, label="speed_kmh"); ax2.set_ylabel("km/h")
    ax.axvline(g.loc[(g.t_rel - t).abs().idxmin(), "t_abs"], color="r", ls="--", lw=.8)
    ax.set_title(f"{rid} t_rel≈{t:.0f}s の前後 30 秒"); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()

上位 10 窓はすべて ride_004（往路）の 14:05〜14:08 に集まっており、20〜40 km/h で rms 1.5〜2.6 g。z は 1.7〜2.1 で、しきい値 3 には届かない。
上位の窓に心当たり（段差・マンホール・踏切・工事区間・荒れた舗装など）があるかは、走った人に確かめる。

## 補足：Isolation Forest のスコア

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for rid, g in M.groupby("ride_id"):
    if rid in ("ride_004", "ride_005", "ride_007", "ride_009"):
        axes[0].hist(g.score_if.dropna(), bins=60, histtype="step", density=True, label=rid)
axes[0].axvline(M.loc[M.is_anomaly_if, "score_if"].min(), color="r", ls="--", lw=.8, label="判定しきい値（contamination 3%）")
axes[0].set_xlabel("score_if（大きいほど異常）"); axes[0].legend()
axes[1].scatter(MV.z_speed, MV.score_if, s=6, alpha=.5); axes[1].set_xlabel("z_speed"); axes[1].set_ylabel("score_if")
rho, _ = spearmanr(MV.z_speed, MV.score_if); axes[1].set_title(f"走行中の窓：Spearman {rho:.2f}"); axes[1].grid(alpha=.3)
plt.tight_layout()
ia = M[M.is_anomaly_if]
print("IF 異常窓:", len(ia), " 走行別:", ia.ride_id.value_counts().to_dict())
print("IF 異常窓の rms 中央値:", round(ia.rms.median(), 2), "g（全窓", round(M.rms.median(), 2), "g）")

- `score_if` は速度を使わないので、単に「振動が大きい窓」（rms 中央値 5 g 超）を拾う。異常窓の 9 割は郊外走行（ride_007/009）にあり、路面が荒い区間と考えられる。
- `z_speed` との相関はほぼ 0。2 つのスコアは別のものを見ている（速度相対 vs 絶対的な振動の大きさ）。

## 結論

| チェック | 結果 |
|---|---|
| 速度との関係 | 期待どおり正（5 km/h 以上でビン中央値が単調に近く上がる） |
| 往路・復路の一致 | **確認できず**（20〜100 m グリッドで Spearman ≈ 0、共通マスが少ない。速度も一致しない） |
| 上位異常の目視 | 走行中に z>3 はなし。上位 10 窓（z 1.7〜2.1）は ride_004 14:05〜14:08 に集中。心当たりは未確認 |

**画面で見せる主スコアは `z_speed`**（速度で補正した正常ベースラインからの逸脱。ラベルなしでも意味が明確）。`score_if` は補助。